# Notebook 03 — Modelagem

## Tech Challenge Fase 1 — Classificação de SRAG (SIVEP-Gripe)

**Objetivo:** Treinar e otimizar múltiplos modelos de classificação para prever o desfecho clínico (Cura ou Óbito).

---

## Modelos implementados

| # | Modelo | Tipo | Por quê |
|---|--------|------|---------|
| 1 | Regressão Logística | Linear | Baseline simples e interpretável |
| 2 | Árvore de Decisão | Não-linear | Altamente interpretável, sem normalização |
| 3 | Random Forest | Ensemble | Robusto, captura não-linearidades |
| 4 | XGBoost | Gradient Boosting | Geralmente melhor performance |

## Estrutura do notebook
1. Configuração e carregamento dos dados processados
2. Treinamento individual de cada modelo
3. Avaliação no conjunto de validação
4. Comparativo e seleção do melhor modelo
5. Salvamento dos modelos

## 1. Configuração

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.metrics import f1_score, classification_report

warnings.filterwarnings('ignore')

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from src.modeling import definir_modelos, treinar_modelo, treinar_todos_modelos
from src.evaluation import calcular_metricas, comparar_modelos

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_style('whitegrid')
print('Configurado.')

## 2. Carregamento dos Dados Processados

> Execute o Notebook 02 antes para gerar os artefatos em `data/processed/`.

In [ ]:
PROCESSED = ROOT / 'data' / 'processed'

X_train = pd.read_parquet(PROCESSED / 'X_train.parquet')
X_val   = pd.read_parquet(PROCESSED / 'X_val.parquet')
X_test  = pd.read_parquet(PROCESSED / 'X_test.parquet')
y_train = pd.read_parquet(PROCESSED / 'y_train.parquet').squeeze()
y_val   = pd.read_parquet(PROCESSED / 'y_val.parquet').squeeze()
y_test  = pd.read_parquet(PROCESSED / 'X_test.parquet')  # carregado abaixo
y_test  = pd.read_parquet(PROCESSED / 'y_test.parquet').squeeze()

print(f'Treino:    X={X_train.shape}, y={y_train.shape} | Óbito: {y_train.mean()*100:.1f}%')
print(f'Validação: X={X_val.shape}, y={y_val.shape} | Óbito: {y_val.mean()*100:.1f}%')
print(f'Teste:     X={X_test.shape}, y={y_test.shape} | Óbito: {y_test.mean()*100:.1f}%')

## 3. Modelo 1 — Regressão Logística (Baseline)

**Justificativa:** Modelo linear simples, serve como baseline. Coeficientes são interpretáveis diretamente como log-odds. Requer normalização (já aplicada no preprocessing).

In [ ]:
modelos_config = definir_modelos()

# Treino com GridSearchCV (busca de hiperparâmetros)
# Para teste rápido, use usar_grid_search=False
lr = treinar_modelo('logistic_regression', modelos_config['logistic_regression'],
                    X_train, y_train, usar_grid_search=True)

# Avaliação na validação
metricas_lr = calcular_metricas('logistic_regression', lr, X_val, y_val)

In [ ]:
# Visualizar os coeficientes mais importantes
coefs = pd.Series(lr.coef_[0], index=X_train.columns).sort_values(key=abs, ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
cores = ['#F44336' if v > 0 else '#2196F3' for v in coefs.values]
ax.barh(coefs.index[::-1], coefs.values[::-1], color=cores[::-1])
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_title('Coeficientes — Regressão Logística (Top 15)')
ax.set_xlabel('Coeficiente (log-odds)')
plt.tight_layout()
plt.savefig('../results/figures/coeficientes_logistic_regression.png', dpi=150)
plt.show()

## 4. Modelo 2 — Árvore de Decisão

**Justificativa:** Altamente interpretável — é possível visualizar o caminho de decisão. Não requer normalização. Tendência a overfitting sem regularização (controlamos via `max_depth`).

In [ ]:
dt = treinar_modelo('decision_tree', modelos_config['decision_tree'],
                    X_train, y_train, usar_grid_search=True)

metricas_dt = calcular_metricas('decision_tree', dt, X_val, y_val)

print(f'\nProfundidade da árvore treinada: {dt.get_depth()}')
print(f'Número de folhas: {dt.get_n_leaves()}')

In [ ]:
# Feature importance da Árvore
from src.evaluation import plotar_feature_importance
plotar_feature_importance('decision_tree', dt, list(X_train.columns))

## 5. Modelo 3 — Random Forest

**Justificativa:** Ensemble de múltiplas árvores com bagging. Reduz overfitting em relação à árvore única. Robusto a outliers e variáveis irrelevantes. Feature importance via média das impurezas.

In [ ]:
rf = treinar_modelo('random_forest', modelos_config['random_forest'],
                    X_train, y_train, usar_grid_search=True)

metricas_rf = calcular_metricas('random_forest', rf, X_val, y_val)

In [ ]:
plotar_feature_importance('random_forest', rf, list(X_train.columns))

## 6. Modelo 4 — XGBoost

**Justificativa:** Gradient Boosting que constrói árvores sequencialmente, cada uma corrigindo os erros da anterior. Geralmente superior a Random Forest em dados tabulares. Possui regularização L1/L2 nativa.

In [ ]:
xgb = treinar_modelo('xgboost', modelos_config['xgboost'],
                     X_train, y_train, usar_grid_search=True)

metricas_xgb = calcular_metricas('xgboost', xgb, X_val, y_val)

In [ ]:
plotar_feature_importance('xgboost', xgb, list(X_train.columns))

## 7. Comparativo na Validação

In [ ]:
resultados_val = [metricas_lr, metricas_dt, metricas_rf, metricas_xgb]
df_comparativo = comparar_modelos(resultados_val)

# Gráfico comparativo
metricas_plot = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
df_plot = df_comparativo[metricas_plot]

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(df_plot))
width = 0.15
cores = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']

for i, (metrica, cor) in enumerate(zip(metricas_plot, cores)):
    ax.bar(x + i * width, df_plot[metrica], width, label=metrica.upper(), color=cor, alpha=0.85)

ax.set_xlabel('Modelo')
ax.set_ylabel('Score')
ax.set_title('Comparativo de Métricas na Validação')
ax.set_xticks(x + width * 2)
ax.set_xticklabels(df_plot.index, rotation=15)
ax.legend(loc='lower right')
ax.set_ylim(0, 1.1)
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('../results/figures/comparativo_modelos_validacao.png', dpi=150)
plt.show()

## 8. Salvamento dos Modelos

In [ ]:
MODELS_DIR = ROOT / 'results' / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

modelos_treinados = {
    'logistic_regression': lr,
    'decision_tree': dt,
    'random_forest': rf,
    'xgboost': xgb,
}

for nome, modelo in modelos_treinados.items():
    caminho = MODELS_DIR / f'{nome}.pkl'
    joblib.dump(modelo, caminho)
    print(f'Salvo: {caminho}')

print('\nTodos os modelos salvos em results/models/')

## 9. Conclusões da Modelagem

**Análise dos resultados na validação:**

> *Preencha após executar as células acima com os valores reais obtidos.*

**Escolha da métrica principal — F1-score:**

Em um contexto médico de classificação de óbito, priorizamos o **F1-score** como métrica principal porque:
- O dataset é **desbalanceado** (mais curas que óbitos)
- **Falsos negativos** (predizer cura quando é óbito) têm custo clínico alto
- O F1 equilibra Precision e Recall, sendo mais informativo que Accuracy
- Complementado pelo **ROC-AUC** para avaliar o poder discriminativo geral

**Próximos passos:** Notebook 04 — Avaliação e Interpretabilidade (conjunto de TESTE)